# TB Portals — Kantipudi A2 strict baseline replication

Reproduces **approach A2** from Kantipudi et al. (JIIM 2024): two *separate* DenseNet121 models (ALP regressor + cavity classifier) trained on **lung-cropped** 224x224 images, country-segregated test.

Pipeline: clone repo -> build the 5,010-image manifest (Table 1) -> MedSAM lung crops -> train ALP + cavity -> evaluate vs the paper.

**Attach these Kaggle datasets before running:**
- `tb-portals-cxr-pngs` (your August-2023 PNG export)
- `medsam-vit-b` (MedSAM ViT-B checkpoint)

## 0 - Clone the codebase

In [8]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + '/scripts'):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)


Already up to date.
repo ready at /kaggle/working/dl-project-codebase


In [9]:
# MedSAM lung segmentation needs segment-anything; pydicom is a harmless extra.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")


deps installed


## Paths

Edit `DATASET` / `MEDSAM_CKPT` if your dataset slugs differ.

In [10]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
OUT_DIR        = f"{WORK}/checkpoints/paper_baseline"
os.makedirs(OUT_DIR, exist_ok=True)
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))


KAGGLE_EXPORT: /kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs/kaggle_export -> True
MEDSAM_CKPT:   /kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth -> True
LUNG_DECODER:  /kaggle/working/dl-project-codebase/checkpoints/component4/component4_mask_decoder.pt -> True


## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

Subsamples your August-2023 export to the paper's exact per-country, per-cavity counts (fixed seed=42, deterministic).

In [11]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample, PAPER_TOTAL

raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
# relative 'images/x.png' -> absolute Kaggle path
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
print(f"Loaded {len(raw)} August-2023 images")

paper_df = subsample(raw, seed=42)
# image_id must be filesystem-safe (crops are saved as <image_id>.png).
# The raw image_id is a path with '/'; use the unique PNG stem instead.
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"\nPaper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")
print("sample image_id:", paper_df["image_id"].iloc[0])


Loaded 8619 August-2023 images
country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12

Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv
sample image_id: 2.25.55583271624075734928315754662077642162


## 2 - Generate MedSAM lung crops (~25 min on T4)

Segments lungs, crops to the lung bounding box, saves 224x224 PNGs keyed by `image_id`. Idempotent: re-running skips existing crops. **Tip:** after this finishes, the last cell zips `crops/` so you can upload it as a dataset and skip this step next time.

In [12]:
import os, sys
if REPO_DIR + '/scripts' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/scripts')
from cache_lung_crops import main as crops_main

argv = ['--manifest', PAPER_MANIFEST, '--out-dir', CROPS_DIR,
        '--medsam-ckpt', MEDSAM_CKPT, '--size', '224', '--pad', '32']
if os.path.isfile(LUNG_DECODER):
    argv += ['--lung-decoder-ckpt', LUNG_DECODER]
else:
    print('[WARN] fine-tuned lung decoder missing; base MedSAM masks are lower quality')
crops_main(argv)
print('crops ->', CROPS_DIR, '| count:', len(os.listdir(CROPS_DIR)))


[crops] device=cuda:Tesla T4
[crops] all 5010 crops already cached in /kaggle/working/crops; skipping MedSAM.
crops -> /kaggle/working/crops | count: 5010


## 3 - Smoke test (~3 min)
1 country, seed 0, 2 epochs. Confirms ALP + cavity training and eval run end-to-end.

In [13]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from src.training.train_baseline_paper import main as train_main
train_main([
    '--manifest',    PAPER_MANIFEST,
    '--crops-dir',   CROPS_DIR,
    '--out-dir',     f'{WORK}/checkpoints/paper_smoke',
    '--held-outs',   'Romania',
    '--seeds',       '0',
    '--epochs',      '2',
    '--batch-size',  '64',
    '--num-workers', '2',
])


[paper-baseline] device=cuda

===== Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[ALP] train=3832 val=958 test=220
  [ALP] epoch 00 train_mse=0.04010 val_mse=0.03634
  [ALP] epoch 01 train_mse=0.02942 val_mse=0.02467
[CAV] train=2918 val=730 (balanced, cavity+=1824)
  [CAV] epoch 00 train_ce=0.69490 val_ce=0.71971
  [CAV] epoch 01 train_ce=0.60945 val_ce=0.60296
[RESULT] Romania seed=0  ALP_MAE=12.36  cavity_AUC=0.684  Timika_MAE=21.11 (15.08%)  Pearson=0.67

TB Portals Timika comparison vs Kantipudi A2  (ours | their reported)

[Romania]  (n=220, cavity+=143)
  Timika MAE       21.11 |  18.70  (below)   CI95=[18.92, 23.43]
  Timika MAE%      15.08 |  13.36  (below)
  Timika Pearson    0.67 |   0.70  (below)   CI95=[0.59, 0.73]
  ALP MAE          12.36 |  11.86  (below)   CI95=[11.07, 13.65]
  Cavity AUC        0.68 |   0.80  (below)
  Cavity F1         0.70 |   0.81  (below)


[paper-baseline] mean +/- std across 

## 4 - Full paper run (~2-3 h)
3 held-out countries x seeds 0,1,2 x 30 epochs, batch 300 — the paper-faithful run.
If you hit GPU OOM at batch 300, drop to `--batch-size 256`.

In [14]:
from src.training.train_baseline_paper import main as train_main
train_main([
    '--manifest',    PAPER_MANIFEST,
    '--crops-dir',   CROPS_DIR,
    '--out-dir',     OUT_DIR,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '0', '1', '2',
    '--epochs',      '30',
    '--batch-size',  '60',
    '--accum-steps', '5',
    '--num-workers', '2',
])



[paper-baseline] device=cuda

===== Romania  seed=0 =====
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[ALP] train=3832 val=958 test=220
  [ALP] epoch 00 train_mse=0.06088 val_mse=0.04714
  [ALP] epoch 01 train_mse=0.03957 val_mse=0.02690
  [ALP] epoch 02 train_mse=0.03400 val_mse=0.02800
  [ALP] epoch 03 train_mse=0.02855 val_mse=0.02637
  [ALP] epoch 04 train_mse=0.02360 val_mse=0.02826
  [ALP] epoch 05 train_mse=0.02549 val_mse=0.02452
  [ALP] epoch 06 train_mse=0.02099 val_mse=0.02608
  [ALP] epoch 07 train_mse=0.01923 val_mse=0.02317
  [ALP] epoch 08 train_mse=0.01865 val_mse=0.02927
  [ALP] epoch 09 train_mse=0.02055 val_mse=0.03441
  [ALP] epoch 10 train_mse=0.02051 val_mse=0.02453
  [ALP] epoch 11 train_mse=0.01957 val_mse=0.02502
  [ALP] epoch 12 train_mse=0.01633 val_mse=0.02970
  [ALP] epoch 13 train_mse=0.01547 val_mse=0.02364
  [ALP] epoch 14 train_mse=0.01586 val_mse=0.02603
  [ALP] epoch 15 train_mse=0.01477 val_mse=0.023

## 5 - Compare to the paper (Table 6/7)

In [15]:
import pandas as pd
res = pd.read_csv(f'{OUT_DIR}/results.csv')
print(res.to_string())

# Kantipudi A2 targets: (ALP_MAE, cavity_AUC, Timika_MAE, Timika_Pearson)
KANTIPUDI = {'Romania': (11.86, 0.80, 18.70, 0.70),
             'Moldova': (16.24, 0.88, 18.85, 0.84),
             'Kazakhstan': (12.16, 0.85, 19.62, 0.70)}
print()
print(f"{'country':12s} {'ALP_MAE ours|paper':>20s} {'cavAUC ours|paper':>20s} {'Timika ours|paper':>20s}")
for ho, (alp_p, auc_p, tm_p, _pear) in KANTIPUDI.items():
    s = res[res.held_out == ho]
    if len(s) == 0:
        continue
    print(f"{ho:12s}   {s.alp_mae.mean():6.2f} | {alp_p:5.2f}      "
          f"{s.cavity_auc.mean():.3f} | {auc_p:.2f}      "
          f"{s.timika_mae.mean():6.2f} | {tm_p:5.2f}")


     held_out  seed  n_test    alp_mae  cavity_auc  cavity_f1  cavity_precision  cavity_recall  timika_mae  timika_mae_pct  timika_pearson  alp_best_val_mse  cav_best_val_ce
0     Romania     0     220  12.357160    0.719553   0.723881          0.776000       0.678322   20.232775       14.451982        0.656260          0.023167         0.556775
1     Romania     1     220  12.011095    0.735174   0.756364          0.787879       0.727273   19.699819       14.071299        0.675135          0.021527         0.520783
2     Romania     2     220  13.091295    0.710017   0.761566          0.775362       0.748252   19.867039       14.190742        0.675381          0.024031         0.594799
3     Moldova     0     589  23.559437    0.820053   0.632768          0.695652       0.580311   27.617238       19.726599        0.745255          0.022439         0.559244
4     Moldova     1     589  24.983057    0.799484   0.595870          0.691781       0.523316   30.368579       21.691842        

## 6 - Save outputs (download these)
- `results.zip` — the metrics + exact 5,010-image manifest (small, always grab)
- `lung_crops.zip` — re-upload as a dataset to skip Section 2 next time (~150 MB)
- `checkpoints_paper_baseline.zip` — trained ALP/cavity weights for re-eval or the MoE phase (~0.5 GB)

In [16]:
!cd /kaggle/working && zip -j results.zip checkpoints/paper_baseline/results.csv tbportals_manifest_paper.csv
!cd /kaggle/working && zip -r -q lung_crops.zip crops
!cd /kaggle/working && zip -r -q checkpoints_paper_baseline.zip checkpoints/paper_baseline
print("Saved in /kaggle/working: results.zip, lung_crops.zip, checkpoints_paper_baseline.zip")


  adding: results.csv (deflated 50%)
  adding: tbportals_manifest_paper.csv (deflated 77%)
Saved in /kaggle/working: results.zip, lung_crops.zip, checkpoints_paper_baseline.zip
